In [131]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [132]:
bias_types = ["less_positive_class"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
datasets = ["folktables_employment", "folktables_income", "hr_analytics", "breast_cancer", "loan_prediction"]
methods_list = ["uniform","psa", "kmm", "mrs-forest", "soft-mrs-exponential", "fw-mrs-temperature",  "fw-mrs-temperature-svm"]

In [133]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else: 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                method_path = Path(f"../thesis_results/downstream_task/{dataset}/{bias_type}/{bias_strength}/{method}/\
classification_results/abs_feature_importance.json")
                unbiased_path = Path(f"../thesis_results/downstream_task/{dataset}/none/0.1/uniform/classification_results/\
R_feature_importance_list.json")
                with open(method_path) as file:
                    method_feature_importance = np.array(json.load(file))
                with open(unbiased_path) as file:
                    uniform_feature_importance = np.array(json.load(file))
                dict_list.append(
                    {
                        "Method": method,
                        "Data Set": dataset, 
                        "Feature Importances": method_feature_importance, 
                        "Bias Type": bias_type,
                        "Bias Strength": bias_strength,})
                
                dict_list.append(
                    {
                        "Method": "unbiased",
                        "Data Set": dataset, 
                        "Feature Importances": uniform_feature_importance, 
                        "Bias Type": bias_type,
                        "Bias Strength": bias_strength,})
                
result_df = pd.DataFrame(data=dict_list)

In [134]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in methods_list:
            mean_mean_difference_list = []
            std_mean_difference_list = []
            for dataset in datasets:
                method_importances = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["Feature Importances"].iloc[0]
                unbiased_importances = result_df.loc[(result_df["Method"]=="unbiased") & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["Feature Importances"].iloc[0]
                mean_mean_abs_difference = mean_mean_difference = np.mean(np.sum(np.abs(method_importances - unbiased_importances), axis=1))
                mean_mean_difference_list.append(np.round(mean_mean_abs_difference, 3))
                std_mean_abs_difference = mean_mean_difference = np.std(np.sum(np.abs(method_importances - unbiased_importances), axis=1))
                std_mean_difference_list.append(np.round(std_mean_abs_difference, 3))

            print(f"\t& {method} \
& ${mean_mean_difference_list[0]}\\pm{std_mean_difference_list[0]}$ \
& ${mean_mean_difference_list[1]}\\pm{std_mean_difference_list[1]}$ \
& ${mean_mean_difference_list[2]}\\pm{std_mean_difference_list[2]}$ \
& ${mean_mean_difference_list[3]}\\pm{std_mean_difference_list[3]}$ \
& ${mean_mean_difference_list[4]}\\pm{std_mean_difference_list[4]}$ & \\\\")
        print("\n")

less_positive_class, 0.1
	& uniform & $0.176\pm0.044$ & $0.199\pm0.058$ & $0.166\pm0.065$ & $0.233\pm0.061$ & $0.241\pm0.054$ & \\
	& psa & $0.192\pm0.026$ & $0.242\pm0.054$ & $0.16\pm0.068$ & $0.211\pm0.055$ & $0.301\pm0.057$ & \\
	& kmm & $0.243\pm0.056$ & $0.25\pm0.041$ & $0.192\pm0.089$ & $0.191\pm0.053$ & $0.34\pm0.065$ & \\
	& mrs-forest & $0.173\pm0.025$ & $0.199\pm0.051$ & $0.165\pm0.067$ & $0.217\pm0.049$ & $0.254\pm0.057$ & \\
	& soft-mrs-exponential & $0.196\pm0.038$ & $0.234\pm0.045$ & $0.168\pm0.069$ & $0.196\pm0.051$ & $0.325\pm0.059$ & \\
	& fw-mrs-temperature & $0.245\pm0.033$ & $0.281\pm0.05$ & $0.18\pm0.067$ & $0.319\pm0.121$ & $0.376\pm0.103$ & \\
	& fw-mrs-temperature-svm & $0.219\pm0.032$ & $0.24\pm0.049$ & $0.192\pm0.08$ & $0.324\pm0.123$ & $0.372\pm0.072$ & \\


